# Proyecto Final - Inteligencia Artificial
## Prediccion de Desercion Estudiantil

Este notebook entrena modelos de Machine Learning para detectar estudiantes con riesgo de abandono academico. El flujo cumple con: carga de datos, limpieza, entrenamiento, evaluacion, comparacion de modelos e interpretacion.

## 1. Instalacion opcional
Ejecutar esta celda en Google Colab si se desea descargar el dataset real desde UCI.

In [ ]:
# !pip install ucimlrepo pandas numpy scikit-learn matplotlib

## 2. Importacion de librerias

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

## 3. Carga del dataset
Primero se intenta usar el dataset real de UCI. Si no hay conexion, se puede cargar el CSV de muestra incluido en la carpeta `data`.

In [ ]:
def cargar_dataset():
    try:
        from ucimlrepo import fetch_ucirepo
        dataset = fetch_ucirepo(id=697)
        X = dataset.data.features.copy()
        y = dataset.data.targets.copy()
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]
        df = X.copy()
        df['Target'] = y
        print('Dataset real cargado desde UCI.')
        return df
    except Exception as e:
        print('No se pudo descargar desde UCI. Cargar archivo local si se esta en el repositorio.')
        ruta = Path('../data/student_dropout_sample.csv')
        if ruta.exists():
            return pd.read_csv(ruta)
        return pd.read_csv('data/student_dropout_sample.csv')

df = cargar_dataset()
df.head()

## 4. Analisis exploratorio basico

In [ ]:
print('Dimension del dataset:', df.shape)
print('
Valores nulos por columna:')
print(df.isna().sum().sort_values(ascending=False).head(10))
print('
Distribucion de la variable objetivo:')
print(df['Target'].value_counts())

In [ ]:
df['Target'].value_counts().plot(kind='bar', title='Distribucion de clases')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.tight_layout()
plt.show()

## 5. Preparacion de datos
Se convierte el problema a clasificacion binaria: `Dropout = 1`, `No Dropout = 0`. Esto permite enfocarse directamente en identificar estudiantes en riesgo.

In [ ]:
df_modelo = df.dropna().copy()
df_modelo['Dropout_bin'] = df_modelo['Target'].astype(str).str.lower().eq('dropout').astype(int)

X = df_modelo.drop(columns=['Target', 'Dropout_bin'])
y = df_modelo['Dropout_bin']

for col in X.columns:
    if not pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].astype('category').cat.codes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 6. Entrenamiento de modelos
Se comparan tres modelos: Regresion Logistica, Arbol de Decision y Random Forest.

In [ ]:
modelos = {
    'Regresion Logistica': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ]),
    'Arbol de Decision': DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
}

resultados = []
predicciones = {}
modelos_entrenados = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1] if hasattr(modelo, 'predict_proba') else None
    metricas = {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    }
    resultados.append(metricas)
    predicciones[nombre] = y_pred
    modelos_entrenados[nombre] = modelo
    print('
', nombre)
    print(classification_report(y_test, y_pred, target_names=['No Dropout', 'Dropout'], zero_division=0))

## 7. Comparacion de resultados

In [ ]:
resultados_df = pd.DataFrame(resultados).sort_values(by='F1-score', ascending=False)
resultados_df.round(3)

In [ ]:
mejor_modelo_nombre = resultados_df.iloc[0]['Modelo']
print('Mejor modelo segun F1-score:', mejor_modelo_nombre)
cm = confusion_matrix(y_test, predicciones[mejor_modelo_nombre])
plt.imshow(cm)
plt.title('Matriz de confusion - ' + mejor_modelo_nombre)
plt.xlabel('Prediccion')
plt.ylabel('Valor real')
plt.xticks([0,1], ['No Dropout','Dropout'])
plt.yticks([0,1], ['No Dropout','Dropout'])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha='center', va='center')
plt.colorbar()
plt.tight_layout()
plt.show()

## 8. Importancia de variables
En modelos tipo bosque aleatorio, se puede revisar que variables influyen mas en la prediccion.

In [ ]:
rf = modelos_entrenados['Random Forest']
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
importancias.sort_values().plot(kind='barh', title='Top 10 variables mas importantes')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

## 9. Prediccion de ejemplo

In [ ]:
estudiante = X_test.iloc[[0]]
prediccion = modelos_entrenados[mejor_modelo_nombre].predict(estudiante)[0]
print('Resultado:', 'Riesgo de desercion' if prediccion == 1 else 'Sin riesgo alto')

## 10. Conclusiones
- La desercion estudiantil puede modelarse como un problema de clasificacion supervisada.
- El sistema no debe usarse para castigar o etiquetar estudiantes, sino para activar apoyo academico y financiero.
- Las metricas mas importantes son recall y F1-score, porque interesa detectar la mayor cantidad posible de estudiantes en riesgo sin descuidar los falsos positivos.